# Лабораторна робота 2 — Проста лінійна регресія (аналітичний розв'язок)


**Набір даних:** `kc_house_data.csv`  
**Обмеження:** scikit-learn-регресія **не дозволена** для базових завдань.

## Налаштування

In [ ]:
from google.colab import drive
import sys
drive.mount('/content/drive')
sys.path.append('/content/drive/MyDrive/MTAD_Labs/Lab2')

Mounted at /content/drive


In [ ]:
import sys
!{sys.executable} -m pip install numpy pandas matplotlib --quiet


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

%matplotlib inline


## Теоретичне підґрунтя

Для однієї вхідної ознаки **x** та виходу **y** аналітичний розв'язок МНК:
```
slope     = ( Σ(xᵢ·yᵢ) − n·x̄·ȳ )  /  ( Σ(xᵢ²) − n·x̄² )
intercept = ȳ − slope · x̄
```
Сума квадратів залишків (RSS):
```
RSS = Σ ( yᵢ − (intercept + slope·xᵢ) )²
```

---
## Завдання 1 — Підготовка даних

Завантажте `kc_house_data.csv`. Розбийте на **навчальну (20 %) та тестову (80 %)** вибірки з `random_state=0`.

In [ ]:
sales = pd.read_csv('/content/drive/MyDrive/MTAD_Labs/Lab2/kc_house_data.csv')
train_data, test_data = train_test_split(sales, test_size=0.8, random_state=0)
print(f'Навчальна: {len(train_data)} рядків  |  Тестова: {len(test_data)} рядків')


Навчальна: 4322 рядків  |  Тестова: 17291 рядків


---
## Завдання 2 — Реалізація `simple_linear_regression()`

Завершіть функцію, використовуючи **лише NumPy** (без sklearn, без циклів по рядках).

In [ ]:
def simple_linear_regression(input_feature, output):
    """
    Обчислює slope та intercept МНК для однієї ознаки.

    Параметри
    ----------
    input_feature : array-like, shape (n,)
    output        : array-like, shape (n,)

    Повертає
    -------
    intercept, slope : float, float
    """
    input_feature = np.array(input_feature, dtype=float)
    output        = np.array(output, dtype=float)
    n = len(input_feature)

    x_mean = input_feature.mean()
    y_mean = output.mean()

    sum_xy = (input_feature * output).sum()
    numerator = sum_xy - n * x_mean * y_mean

    sum_xx = (input_feature**2).sum()
    denominator = sum_xx - n * (x_mean**2)

    slope = numerator / denominator
    intercept = y_mean - slope * x_mean

    return intercept, slope


### Перевірка — обидва значення нижче мають дорівнювати **1.0**

In [ ]:
test_feature = np.arange(5, dtype=float)
test_output  = 1.0 + 1.0 * test_feature
test_intercept, test_slope = simple_linear_regression(test_feature, test_output)
print(f'Intercept: {test_intercept:.4f}  (очікується 1.0)')
print(f'Slope    : {test_slope:.4f}  (очікується 1.0)')


Intercept: 1.0000  (очікується 1.0)
Slope    : 1.0000  (очікується 1.0)


### Навчання на `sqft_living`

In [ ]:
sqft_intercept, sqft_slope = simple_linear_regression(
    train_data['sqft_living'], train_data['price']
)
print(f'Вільний член: {sqft_intercept:.2f}')
print(f'Нахил: {sqft_slope:.4f}')


Вільний член: -29748.29
Нахил: 273.5299


---
## Завдання 3 — Передбачення та RSS

**а)** Реалізуйте `get_regression_predictions(input_feature, intercept, slope)` — повертає масив NumPy передбачених значень.  
**б)** Реалізуйте `get_residual_sum_of_squares(input_feature, output, intercept, slope)` — обчислює RSS.  
**в)** Перевірте обидві функції, потім вкажіть RSS на навчальній і тестовій вибірках та дайте відповідь на питання нижче.

In [ ]:
def get_regression_predictions(input_feature, intercept, slope):
    """Повертає масив передбачених значень."""
    predicted_values = intercept + (slope * input_feature)
    return predicted_values


In [ ]:
def get_residual_sum_of_squares(input_feature, output, intercept, slope):
    """Повертає RSS (скаляр)."""
    predictions = get_regression_predictions(input_feature, intercept, slope)
    residuals = output - predictions
    RSS = (residuals**2).sum()
    return RSS


### Перевірка — RSS на тестових вхідних даних має бути **0.0**

In [ ]:
rss_check = get_residual_sum_of_squares(
    test_feature, test_output, test_intercept, test_slope
)
print(f'RSS на тестових вхідних даних: {rss_check:.2f}  (очікується 0.0)')


RSS на тестових вхідних даних: 0.00  (очікується 0.0)


### RSS для моделі `sqft_living`

In [ ]:
rss_train = get_residual_sum_of_squares(
    train_data['sqft_living'], train_data['price'],
    sqft_intercept, sqft_slope
)
rss_test = get_residual_sum_of_squares(
    test_data['sqft_living'], test_data['price'],
    sqft_intercept, sqft_slope
)
print(f'Навчальна RSS: {rss_train:.2e}')
print(f'Тестова  RSS: {rss_test:.2e}')


Навчальна RSS: 2.74e+14
Тестова  RSS: 1.20e+15


### Питання — яка передбачувана ціна будинку площею 2 650 кв. футів?

In [ ]:
my_house_sqft = 2650
# Завдання 3в — обчисліть і виведіть передбачувану ціну
estimated_price = get_regression_predictions(my_house_sqft, sqft_intercept, sqft_slope)
print(f'Передбачувана ціна для {my_house_sqft} кв. футів: ${estimated_price:,.2f}')

Передбачувана ціна для 2650 кв. футів: $695,105.98


---
## Завдання 4 — Порівняння двох ознак

Навчіть другу модель, використовуючи `bedrooms` як вхідну ознаку. Обчисліть RSS на **тестовій вибірці** для обох моделей (`sqft_living` і `bedrooms`). Яка ознака дає кращий прогноз? Поясніть у 2–3 реченнях.

In [ ]:
# Навчіть модель на ознаці bedrooms
bed_intercept, bed_slope = simple_linear_regression(
    train_data['bedrooms'], train_data['price']
)
# Обчисліть тестову RSS для обох моделей
rss_test_bedrooms = get_residual_sum_of_squares(
    test_data['bedrooms'], test_data['price'], bed_intercept, bed_slope
)

print(f'RSS для моделі (sqft_living): {rss_test:.2e}')
print(f'RSS для моделі (bedrooms):    {rss_test_bedrooms:.2e}')

RSS для моделі (sqft_living): 1.20e+15
RSS для моделі (bedrooms):    2.15e+15


**Відповідь:** Модель на основі площі (sqft_living) є кращою, оскільки вона має значно менше значення RSS на тестовій вибірці порівняно з моделлю на основі кількості спалень (bedrooms). Це пояснюється тим, що загальна площа будинку є більш точним показником його розміру та вартості, тоді як кількість спалень є дискретним числом і не враховує реальні габарити приміщень.